In [1]:
from __future__ import annotations

from pathlib import Path
import duckdb


DDL = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    full_name   VARCHAR NOT NULL,
    email       VARCHAR NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS orders (
    order_id    INTEGER PRIMARY KEY,
    order_date  DATE NOT NULL,
    total_amount DOUBLE NOT NULL CHECK (total_amount >= 0)
);

-- Bridge table (allows many-to-many; also demonstrates FK integrity cleanly)
CREATE TABLE IF NOT EXISTS customer_orders (
    customer_id INTEGER NOT NULL,
    order_id    INTEGER NOT NULL,
    PRIMARY KEY (customer_id, order_id),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);
"""


def create_database(target_folder: str | Path, db_filename: str = "shop.duckdb") -> Path:
    target_folder = Path(target_folder)
    target_folder.mkdir(parents=True, exist_ok=True)

    db_path = target_folder / db_filename

    con = duckdb.connect(str(db_path))  # file path -> persistent DB file :contentReference[oaicite:2]{index=2}
    try:
        con.execute(DDL)

        # Clean re-run behavior for demos
        con.execute("DELETE FROM customer_orders;")
        con.execute("DELETE FROM orders;")
        con.execute("DELETE FROM customers;")

        # Insert parent tables first (so FKs can be satisfied)
        con.execute("""
            INSERT INTO customers (customer_id, full_name, email) VALUES
            (1, 'Aster Ng',  'aster@example.com'),
            (2, 'Beryl Kim', 'beryl@example.com'),
            (3, 'Cobalt Li', 'cobalt@example.com');
        """)

        con.execute("""
            INSERT INTO orders (order_id, order_date, total_amount) VALUES
            (101, '2025-01-03', 120.50),
            (102, '2025-01-10',  75.00),
            (103, '2025-02-01',  42.25),
            (104, '2025-02-04', 310.00);
        """)

        # Insert bridge table rows (must reference existing customers/orders)
        con.execute("""
            INSERT INTO customer_orders (customer_id, order_id) VALUES
            (1, 101),
            (2, 102),
            (1, 103),
            (3, 104);
        """)

        # Optional integrity self-test: should fail (FK violation)
        try:
            con.execute("INSERT INTO customer_orders (customer_id, order_id) VALUES (999, 101);")
            raise RuntimeError("Expected FK violation did not occur.")
        except Exception:
            # Expected: foreign key constraint violation
            pass

    finally:
        con.close()

    return db_path


db_file = create_database("./database/", "shop.duckdb")
print(f"Created DuckDB database at: {db_file.resolve()}")

Created DuckDB database at: /media/SN850X-A/vscode/kaggle/garvis/langgraph-lab/03. duck-db-integration/database/shop.duckdb
